In [0]:
from pyspark.sql.functions import col

def join_code_desc(silver_df, code_table_name):
  # Define fully qualified table names
  
  code_full_path = f"workspace.code_desc_project_master.{code_table_name}"

  # Read the tables to check columns
  
  code_df = spark.table(code_full_path)

  join_column, add_columns = code_df.columns[0], code_df.columns[1:]

  if join_column in silver_df.columns:
      
      try: 
        # Perform join (will have both join columns)
        result_df = silver_df.alias("stn").join(
        code_df.alias("cdt"),
        on=col(f"stn.{join_column}") == col(f"cdt.{join_column}"),
        how="left").select(
          *[col(f"stn.{c}") for c in silver_df.columns],  # all silver cols with explicit alias
          *[col(f"cdt.{c}") for c in add_columns])

        print(f"Joined {code_table_name} tables")
        print(f"Result row counts: {result_df.count()}, column counts: {len(result_df.columns)}")
        # display(result_df)
        return result_df

      except Exception as e:
          print(f"Failed to join: {e}")
  else:
      print(f"Join column '{join_column}' not found in {table_name} table")

In [0]:
from pyspark.sql.functions import col, to_date

# Create Silver schema to save the silver layer data
spark.sql("CREATE SCHEMA IF NOT EXISTS 2026_january_balance.silver_fund_data")

# Get list of all funds from bronze tables
bronze_tables = spark.sql("SHOW TABLES IN 2026_january_balance.bronze_fund_data").toPandas()
# parse out fund names from bronze tables to a list
fund_names = sorted(set(name.split("_")[0] for name in bronze_tables["tableName"]))


for fund in fund_names:
    print(f"processing {fund}")

    balance = spark.table(f"2026_january_balance.bronze_fund_data.{fund}_january_2026_balances")
    balance = balance.withColumn("Effective_Date", to_date(col("Effective_Date"), "yyyyMMdd"))
    balance = balance.withColumn("Amount", col("Amount").cast("double"))

    print(f"{fund} balance table has {balance.count()} records")

    # Start joining tables together...
    master = balance            #set balance the master table

    # Join accounts table
    try:
        accounts = spark.table(f"2026_january_balance.bronze_fund_data.{fund}_january_2026_accounts")
        # print(f"{fund} accounts table has {accounts.count()} records")
        accounts_cols = [col for col in accounts.columns if col not in balance.columns or col in ['Account_Code', 'Entity_Code']]
        master = master.join(accounts.select(*accounts_cols), on=['Account_Code', 'Entity_Code'], how="left")
        print(f"Done joining accounts table for {fund}")

    except:
        print(f"{fund} accounts table not found")


    # Join departments table - parent department descriptions are available so pull them in
    try:
        departments = spark.table(f"2026_january_balance.bronze_fund_data.{fund}_january_2026_departments")
        print(f"{fund} departments table has {departments.count()} records")
        
        # Get the parent department descriptions
        dept_parent = departments.select(col("Entity_Code"), col("Department_Code"), col("Department_Description").alias("Parent_Dept_Description")) 
        departments_w_parent = departments.join(dept_parent, on =
                                        (departments.Entity_Code == dept_parent.Entity_Code) &
                                        (departments.Parent_Department_Code  == dept_parent.Department_Code),
                                        how="left").drop(dept_parent.Entity_Code, dept_parent.Department_Code)
        
        # Now join with master
        departments_cols = [col for col in departments_w_parent.columns if col not in balance.columns or col in ['Department_Code', 'Entity_Code']]
        master = master.join(departments_w_parent.select(*departments_cols), on=['Department_Code', 'Entity_Code'], how="left")
        print(f"Done joining departments table for {fund}")

    except:
        print(f"{fund} departments table not found")


    # Join funds table. - parent fund codes are not of the same type.
    try:
        funds_data = spark.table(f"2026_january_balance.bronze_fund_data.{fund}_january_2026_funds")
        # print(f"{fund} funds table has {funds_data.count()} records")
        funds_cols = [col for col in funds_data.columns if col not in balance.columns or col in ['Fund_Code', 'Entity_Code']]
        master = master.join(funds_data.select(*funds_cols), on=['Fund_Code', 'Entity_Code'], how="left")
        print(f"Done joining funds table for {fund}")

    except:
        print(f"{fund} funds table not found")

    # Join projects table
    try:
        projects = spark.table(f"2026_january_balance.bronze_fund_data.{fund}_january_2026_projects")
        # print(f"{fund} projects table has {projects.count()} records")
        projects = projects.withColumn("Project_Start_Date", to_date(col("Project_Start_Date"), "yyyyMMdd")).withColumn     ("Project_End_Date", to_date(col("Project_End_Date"), "yyyyMMdd"))

        project_cols = [col for col in projects.columns if col not in balance.columns or col in ['Project_Code', 'Entity_Code']]

        master = master.join(projects.select(*project_cols), on=['Project_Code', 'Entity_Code'], how="left")
        print(f"Done joining projects table for {fund}")

    except:
        print(f"{fund} projects table not found")

    master = master.drop("source_file")

    # Add parent account code description since the join
    pac = spark.table("workspace.code_desc_project_master.parent_account_codes")

    master_parent = master.join(
        pac.select(
            col("Parent_Account_Code"),
            col("Account_Code_Description").alias("Parent_Account_Code_Description"),
            col("Account_type").alias("Parent_Account_Code_Type")
        ),
        on="Parent_Account_Code",
        how="left"
    )

    entity_code_table = "entity_code_desc" 
    entity_code_add = join_code_desc(silver_df=master_parent, code_table_name=entity_code_table)

    function_code_table = "function_code_desc"
    function_code_add = join_code_desc(silver_df=entity_code_add, code_table_name=function_code_table)

    program_code_table = "program_code_desc"
    program_code_add = join_code_desc(silver_df=function_code_add, code_table_name=program_code_table)

    mastertable_name = f"2026_january_balance.silver_fund_data.{fund}_january_2026_master"

    program_code_add.write.format("delta").mode("overwrite").\
                option("OverwriteSchema", "true").\
                saveAsTable(mastertable_name)
    print(f"Done writing {fund} to silver table as {mastertable_name}")    

print("Finished processing all funds")

In [0]:
from pyspark.sql.functions import col, to_date

# Create Silver schema to save the silver layer data
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver_fund_data")

# Get list of all funds from bronze tables
bronze_tables = spark.sql("SHOW TABLES IN workspace.bronze_fund_data").toPandas()
# parse out fund names from bronze tables to a list
fund_names = sorted(set(name.split("_")[0] for name in bronze_tables["tableName"]))


for fund in fund_names:
    print(f"processing {fund}")

    try:
        
        balance = spark.table(f"workspace.bronze_fund_data.{fund}_balances")
        balance = balance.withColumn("Effective_Date", to_date(col("Effective_Date"), "yyyyMMdd"))
        balance = balance.withColumn("Amount", col("Amount").cast("double"))
        
        print(f"{fund} balance table has {balance.count()} records")
        
        # Start joining tables together...
        master = balance            #set balance the master table

        # Join accounts table
        try:
            accounts = spark.table(f"workspace.bronze_fund_data.{fund}_accounts")
            print(f"{fund} accounts table has {accounts.count()} records")
            accounts_cols = [col for col in accounts.columns if col not in balance.columns or col in ['Account_Code', 'Entity_Code']]
            master = master.join(accounts.select(*accounts_cols), on=['Account_Code', 'Entity_Code'], how="left")
            print(f"Done joining accounts table for {fund}")

        except:
            print(f"{fund} accounts table not found")


        # Join departments table - parent department descriptions are available so pull them in
        try:
            departments = spark.table(f"workspace.bronze_fund_data.{fund}_departments")
            print(f"{fund} departments table has {departments.count()} records")
            
            # Get the parent department descriptions
            dept_parent = departments.select(col("Entity_Code"), col("Department_Code"), col("Department_Description").alias("Parent_Dept_Description")) 
            departments_w_parent = departments.join(dept_parent, on =
                                            (departments.Entity_Code == dept_parent.Entity_Code) &
                                            (departments.Parent_Department_Code  == dept_parent.Department_Code),
                                            how="left").drop(dept_parent.Entity_Code, dept_parent.Department_Code)
            
            # Now join with master
            departments_cols = [col for col in departments_w_parent.columns if col not in balance.columns or col in ['Department_Code', 'Entity_Code']]
            master = master.join(departments_w_parent.select(*departments_cols), on=['Department_Code', 'Entity_Code'], how="left")
            print(f"Done joining departments table for {fund}")

        except:
            print(f"{fund} departments table not found")

        
        # Join funds table. - parent fund codes are not of the same type.
        try:
            funds_data = spark.table(f"workspace.bronze_fund_data.{fund}_funds")
            print(f"{fund} funds table has {funds_data.count()} records")
            funds_cols = [col for col in funds_data.columns if col not in balance.columns or col in ['Fund_Code', 'Entity_Code']]
            master = master.join(funds_data.select(*funds_cols), on=['Fund_Code', 'Entity_Code'], how="left")
            print(f"Done joining funds table for {fund}")

        except:
            print(f"{fund} funds table not found")
        
        # Join projects table
        try:
            projects = spark.table(f"workspace.bronze_fund_data.{fund}_projects")
            print(f"{fund} projects table has {projects.count()} records")
            projects = projects.withColumn("Project_Start_Date", to_date(col("Project_Start_Date"), "yyyyMMdd")).withColumn     ("Project_End_Date", to_date(col("Project_End_Date"), "yyyyMMdd"))

            project_cols = [col for col in projects.columns if col not in balance.columns or col in ['Project_Code', 'Entity_Code']]

            master = master.join(projects.select(*project_cols), on=['Project_Code', 'Entity_Code'], how="left")
            print(f"Done joining projects table for {fund}")

        except:
            print(f"{fund} projects table not found")


        # Save master table
        master_table_name = f"workspace.silver_fund_data.{fund}_master"
        master.write.format("delta") \
              .mode("overwrite") \
              .option("overwriteSchema", "true") \
              .saveAsTable(master_table_name)

        print(f"Created {master_table_name} table")

    except Exception as e:
        print(f"Failed to access {fund}: {e}")


print("Silver layer created and uploaded")
            

        

In [0]:
from pyspark.sql.functions import col, to_date

fund= "140a"

balance = spark.table(f"2026_january_balance.bronze_fund_data.{fund}_january_2026_balances")
balance = balance.withColumn("Effective_Date", to_date(col("Effective_Date"), "yyyyMMdd"))
balance = balance.withColumn("Amount", col("Amount").cast("double"))

# print(f"{fund} balance table has {balance.count()} records")

# Start joining tables together...
master = balance            #set balance the master table

# Join accounts table
try:
    accounts = spark.table(f"2026_january_balance.bronze_fund_data.{fund}_january_2026_accounts")
    # print(f"{fund} accounts table has {accounts.count()} records")
    accounts_cols = [col for col in accounts.columns if col not in balance.columns or col in ['Account_Code', 'Entity_Code']]
    master = master.join(accounts.select(*accounts_cols), on=['Account_Code', 'Entity_Code'], how="left")
    print(f"Done joining accounts table for {fund}")

except:
    print(f"{fund} accounts table not found")


# Join departments table - parent department descriptions are available so pull them in
try:
    departments = spark.table(f"2026_january_balance.bronze_fund_data.{fund}_january_2026_departments")
    print(f"{fund} departments table has {departments.count()} records")
    
    # Get the parent department descriptions
    dept_parent = departments.select(col("Entity_Code"), col("Department_Code"), col("Department_Description").alias("Parent_Dept_Description")) 
    departments_w_parent = departments.join(dept_parent, on =
                                    (departments.Entity_Code == dept_parent.Entity_Code) &
                                    (departments.Parent_Department_Code  == dept_parent.Department_Code),
                                    how="left").drop(dept_parent.Entity_Code, dept_parent.Department_Code)
    
    # Now join with master
    departments_cols = [col for col in departments_w_parent.columns if col not in balance.columns or col in ['Department_Code', 'Entity_Code']]
    master = master.join(departments_w_parent.select(*departments_cols), on=['Department_Code', 'Entity_Code'], how="left")
    print(f"Done joining departments table for {fund}")

except:
    print(f"{fund} departments table not found")


# Join funds table. - parent fund codes are not of the same type.
try:
    funds_data = spark.table(f"2026_january_balance.bronze_fund_data.{fund}_january_2026_funds")
    # print(f"{fund} funds table has {funds_data.count()} records")
    funds_cols = [col for col in funds_data.columns if col not in balance.columns or col in ['Fund_Code', 'Entity_Code']]
    master = master.join(funds_data.select(*funds_cols), on=['Fund_Code', 'Entity_Code'], how="left")
    print(f"Done joining funds table for {fund}")

except:
    print(f"{fund} funds table not found")

# Join projects table
try:
    projects = spark.table(f"2026_january_balance.bronze_fund_data.{fund}_january_2026_projects")
    # print(f"{fund} projects table has {projects.count()} records")
    projects = projects.withColumn("Project_Start_Date", to_date(col("Project_Start_Date"), "yyyyMMdd")).withColumn     ("Project_End_Date", to_date(col("Project_End_Date"), "yyyyMMdd"))

    project_cols = [col for col in projects.columns if col not in balance.columns or col in ['Project_Code', 'Entity_Code']]

    master = master.join(projects.select(*project_cols), on=['Project_Code', 'Entity_Code'], how="left")
    print(f"Done joining projects table for {fund}")

except:
    print(f"{fund} projects table not found")

master = master.drop("source_file")

# Add parent account code description since the join
pac = spark.table("workspace.code_desc_project_master.parent_account_codes")

master_parent = master.join(
    pac.select(
        col("Parent_Account_Code"),
        col("Account_Code_Description").alias("Parent_Account_Code_Description"),
        col("Account_type").alias("Parent_Account_Code_Type")
    ),
    on="Parent_Account_Code",
    how="left"
)

entity_code_table = "entity_code_desc" 
entity_code_add = join_code_desc(silver_df=master_parent, code_table_name=entity_code_table)

function_code_table = "function_code_desc"
function_code_add = join_code_desc(silver_df=entity_code_add, code_table_name=function_code_table)

program_code_table = "program_code_desc"
program_code_add = join_code_desc(silver_df=function_code_add, code_table_name=program_code_table)

# enhanced_table_name = f"workspace.silver_fund_data_enhanced.{table_name}_enhanced"
print(f"program_code_add has {program_code_add.count()} records")



In [0]:
program_code_add.createOrReplaceTempView("UCLA_january_master")

display(spark.sql("SELECT * FROM UCLA_january_master"))

In [0]:
# Install openpyxl if needed
%pip install openpyxl

# If you still encounter OSError, restart the kernel with %restart_python before importing the package.

In [0]:
%restart_python

In [0]:
program_code_add.toPandas().to_excel("/Volumes/2026_january_balance/default/jan_2026_raw_data/Davis_Jan_2026_master.xlsx", index=False)